# Detección de patrones y valores atípicos en el mercado inmobiliario de Melbourne mediante Machine Learning no supervisado

**Actividad 1 — Proyecto ETL y EDA**
607526/INTRODUCCION A MACHINE LEARNING

**Integrantes:**
- EDWIN SEBASTIAN RINCON RAMIREZ
- SERGIO ESTEBAN BELLO GOMEZ

**Programa:** Ingeniería de Sistemas y Computación 2024
**Fecha de entrega:** 04/09/2026

---


## 1. Título

**Detección de patrones y valores atípicos en el mercado inmobiliario de Melbourne mediante Machine Learning no supervisado**


## 2. Objetivo general

Analizar el comportamiento del mercado inmobiliario de Melbourne mediante un proceso de extracción,
exploración y preprocesamiento de datos, que permita identificar la calidad inicial del dataset y sentar
las bases para, en una fase posterior (Actividad 2), detectar observaciones atípicas y patrones relevantes
que apoyen la toma de decisiones inmobiliarias.


## 3. Objetivos específicos

1. Extraer y describir la estructura del dataset de vivienda de Melbourne, identificando su calidad inicial
   (valores nulos, registros duplicados, tipos de datos) sin alterar el conjunto de datos original.
2. Realizar un análisis exploratorio de datos (EDA) que revele la distribución, dispersión y relaciones
   entre las variables numéricas (precio, distancia, tamaño del terreno, área construida) y categóricas
   (tipo de propiedad, región, método de venta) más relevantes para el problema.
3. Identificar visualmente posibles valores atípicos y patrones geográficos (mediante latitud y longitud)
   que sustenten el proceso de preprocesamiento, escalamiento y detección de anomalías de la Actividad 2.


## 4. Problemática

El mercado inmobiliario de Melbourne se caracteriza por una alta rotación de propiedades y una fuerte
variabilidad de precios entre suburbios, tipos de vivienda y métodos de venta. Agencias inmobiliarias,
compradores y entidades de planeación urbana necesitan comprender estas dinámicas para tomar
decisiones informadas, pero los registros de transacciones suelen presentar información incompleta
(por ejemplo, área construida o año de construcción sin diligenciar) y valores extremos que pueden
corresponder tanto a errores de captura como a propiedades genuinamente atípicas (de lujo, en muy mal
estado, o con condiciones excepcionales de ubicación).

Sin un proceso sistemático de extracción, exploración y limpieza de datos, estas inconsistencias no se
detectan a tiempo: se toman decisiones sobre información parcial, se entrenan modelos predictivos sobre
datos sin depurar y se pierde la oportunidad de identificar tanto errores como observaciones
verdaderamente interesantes (propiedades sub o sobrevaloradas). Este proyecto aborda esa problemática
construyendo, en dos fases, un pipeline reproducible de preparación de datos: primero mediante un
diagnóstico ETL y EDA (Actividad 1), y después mediante preprocesamiento, escalamiento y detección
de anomalías con métodos estadísticos y de aprendizaje no supervisado (Actividad 2).


## 5. Tecnologías

- **Lenguaje:** Python 3.12
- **Entorno de ejecución:** Jupyter Notebook / JupyterLab
- **Manipulación de datos:** Pandas, NumPy
- **Visualización:** Matplotlib, Seaborn
- **Preprocesamiento y detección de anomalías (Actividad 2):** Scikit-learn, SciPy


## 6. Propuesta de solución

Se implementará un pipeline en Python sobre Jupyter Notebook que, en esta primera fase, extraiga el
dataset *Melbourne Housing Snapshot* y realice un diagnóstico de calidad de datos junto con un análisis
exploratorio (EDA) completamente documentado en Markdown. Los hallazgos de esta fase (variables con
valores nulos, distribución de precios, relaciones entre variables, indicios visuales de outliers y patrones
geográficos) serán el insumo directo de la Actividad 2, en la que se aplicarán técnicas de imputación,
normalización/estandarización y detección de anomalías (IQR, Z-score, Isolation Forest y DBSCAN)
sobre las mismas variables aquí exploradas.


## 7. Extracción inicial de los datos

**Fuente de los datos:** *Melbourne Housing Snapshot* (Kaggle, usuario `dansbecker`,
<https://www.kaggle.com/datasets/dansbecker/melbourne-housing-snapshot>). Es una instantánea estática
creada en septiembre de 2017 a partir de datos de venta de vivienda publicados semanalmente en
Domain.com.au (dataset original recopilado por Tony Pino). Contiene **21 variables** y aproximadamente
**13.580 registros** (se excluyeron del snapshot las propiedades sin precio registrado).

**Cómo obtener el archivo:**
1. Ingresa al enlace del dataset en Kaggle (requiere una cuenta gratuita).
2. Descarga el archivo `melb_data.csv`.
3. Colócalo en una carpeta `data/` en la misma ubicación de este notebook (o ajusta la ruta en la celda
   siguiente).

*Alternativa reproducible:* si tienes configurada la API de Kaggle (`pip install kagglehub`), puedes
descargarlo por código con `kagglehub.dataset_download("dansbecker/melbourne-housing-snapshot")`.

A continuación se importan las librerías necesarias para todo el notebook.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


Cargamos el dataset y conservamos una copia intacta (`df_raw`) que nunca se modificará, para poder
comparar en cualquier momento contra el estado original de los datos.


In [ ]:
# Ajusta la ruta si tu archivo está en otra ubicación
df = pd.read_csv("data/melb_data.csv")

# Copia del dataset original, sin modificaciones, como referencia
df_raw = df.copy()

print("Dataset cargado correctamente.")


Mostramos una muestra inicial de los datos para reconocer visualmente las columnas y el tipo de
información que contiene cada una.


In [ ]:
df.head(10)


Verificamos el número de registros y variables, y el tipo de dato asignado a cada columna. Esto nos
permite distinguir de entrada qué variables son numéricas, cuáles categóricas (`object`) y cuáles podrían
necesitar una conversión de tipo (por ejemplo, `Date`).


In [ ]:
print(f"Número de registros: {df.shape[0]}")
print(f"Número de variables:  {df.shape[1]}")
print()
df.dtypes


Diagnosticamos la presencia de valores nulos por variable (cantidad y porcentaje) y de registros
duplicados. Este resultado es la base directa del diagnóstico de nulos que se profundizará en la
Actividad 2.


In [ ]:
null_summary = df.isna().sum().to_frame("nulos")
null_summary["porcentaje"] = (null_summary["nulos"] / len(df) * 100).round(2)
null_summary = null_summary.sort_values("porcentaje", ascending=False)
display(null_summary)

print(f"\nRegistros duplicados: {df.duplicated().sum()}")


> **Nota de interpretación (completar tras ejecutar):** indica aquí qué variables concentran la mayor
> cantidad de nulos, si tiene sentido imputarlas o no (por ejemplo, un identificador no debería imputarse),
> y si el número de duplicados es significativo frente al tamaño del dataset.


## 8. Análisis Exploratorio de Datos (EDA) inicial

### 8.1 Estadísticos descriptivos de las variables numéricas


In [ ]:
df.describe().T


### 8.2 Frecuencias de las variables categóricas relevantes

Revisamos la distribución de `Type` (tipo de propiedad: h = house, u = unit/dúplex, t = townhouse),
`Method` (método de venta), `Regionname` (región) y `CouncilArea` (municipio), incluyendo los valores
nulos en el conteo (`dropna=False`) para no ocultarlos.


In [ ]:
for col in ["Type", "Method", "Regionname", "CouncilArea"]:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False))
    print()


### 8.3 Distribución de las variables numéricas clave

Histogramas de `Price`, `Landsize`, `BuildingArea` y `Distance`. Se usa `dropna()` en cada variable para
que los valores nulos no interfieran en la visualización (el tratamiento formal de esos nulos se hace en
la Actividad 2).


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
variables = ["Price", "Landsize", "BuildingArea", "Distance"]

for ax, var in zip(axes.flatten(), variables):
    sns.histplot(df[var].dropna(), bins=40, kde=True, ax=ax)
    ax.set_title(f"Distribución de {var}")

plt.tight_layout()
plt.show()


### 8.4 Boxplots para identificar posibles valores atípicos

Boxplot del precio según el tipo de propiedad y según la región. Los puntos fuera de los bigotes son
candidatos visuales a outliers que se formalizarán con IQR y Z-score en la Actividad 2.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=df, x="Type", y="Price", ax=axes[0])
axes[0].set_title("Precio por tipo de propiedad")

sns.boxplot(data=df, x="Regionname", y="Price", ax=axes[1])
axes[1].set_title("Precio por región")
axes[1].tick_params(axis="x", rotation=75)

plt.tight_layout()
plt.show()


### 8.5 Correlación entre variables numéricas


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matriz de correlación de variables numéricas")
plt.tight_layout()
plt.show()


### 8.6 Patrón geográfico (vista previa para DBSCAN en la Actividad 2)

Ubicamos cada propiedad según su latitud y longitud, coloreando por precio. Esta visualización es
particularmente relevante porque `Lattitude` y `Longtitude` serán dos de las variables que usaremos con
DBSCAN en la Actividad 2 para detectar agrupamientos geográficos y puntos de ruido (anomalías
espaciales).


In [ ]:
geo = df.dropna(subset=["Lattitude", "Longtitude", "Price"])

plt.figure(figsize=(9, 7))
sc = plt.scatter(geo["Longtitude"], geo["Lattitude"], c=geo["Price"],
                  cmap="viridis", s=8, alpha=0.6)
plt.colorbar(sc, label="Precio")
plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.title("Distribución geográfica de propiedades según precio")
plt.tight_layout()
plt.show()


## 9. Hallazgos principales (completar tras ejecutar el notebook)

- **Calidad de los datos:** [¿qué variables tienen más nulos? ¿hay duplicados? ¿qué variables no deberían
  imputarse?]
- **Distribución de precios:** [¿la distribución es simétrica o sesgada? ¿hay una cola larga hacia precios
  altos?]
- **Relación tipo de propiedad / precio:** [¿qué tipo de propiedad tiene mayor dispersión o más outliers
  visuales?]
- **Relación región / precio:** [¿qué regiones concentran los precios más altos y más atípicos?]
- **Correlaciones relevantes:** [¿qué variables numéricas están más correlacionadas con `Price`?]
- **Patrón geográfico:** [¿se observan agrupamientos o zonas con precios claramente distintos al resto?]

## 10. Próximos pasos (enlace con la Actividad 2)

A partir de estos hallazgos, en la Actividad 2 se construirá un pipeline de preprocesamiento sobre este
mismo dataset: diagnóstico e imputación formal de los valores nulos identificados en `Car`,
`BuildingArea`, `YearBuilt` y `CouncilArea`; estandarización de las variables numéricas; y detección
comparativa de outliers mediante IQR, Z-score, Isolation Forest y DBSCAN — este último aprovechando
directamente el patrón geográfico explorado en la sección 8.6.
